# 01 — Data Validation and Split Creation

**Objective:** Inspect the RoboMimic HDF5 dataset and create nested subsets: D50, D100, and D200.

Before running: Ensure the notebook is saved inside the `notebooks` directory in Google Drive. All persistent outputs are saved to Drive; only temporary files are kept under `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/DiffusionPolicyEvents')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError('Run notebook 00_Project_Setup first.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.project_paths import ProjectPaths
paths = ProjectPaths(PROJECT_ROOT).ensure()
print(paths)


In [ ]:
# Download the raw Lift Proficient-Human dataset.
# Raw states are required because we will render our own RGB observations
# from both cameras and later generate high-temporal-resolution input for v2e.

%pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil
import hashlib
import h5py

DATASET = paths.raw / "lift_ph_raw.hdf5"
DATASET.parent.mkdir(parents=True, exist_ok=True)

if not DATASET.exists():
    print("Downloading Robomimic Lift PH raw dataset...")

    cached_file = hf_hub_download(
        repo_id="amandlek/robomimic",
        repo_type="dataset",
        filename="v1.5/lift/ph/demo_v15.hdf5",
    )

    shutil.copy2(cached_file, DATASET)
    print(f"Dataset saved to:\n{DATASET}")
else:
    print(f"Dataset already exists:\n{DATASET}")

print(f"File size: {DATASET.stat().st_size / 1024**2:.2f} MB")

In [ ]:
def calculate_sha256(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


with h5py.File(DATASET, "r") as dataset_file:
    demo_ids = sorted(dataset_file["data"].keys())

    print(f"Number of demonstrations: {len(demo_ids)}")
    print(f"First demonstrations: {demo_ids[:5]}")
    print(f"Dataset keys: {list(dataset_file.keys())}")

assert len(demo_ids) == 200, (
    f"Expected 200 Lift PH demonstrations, found {len(demo_ids)}"
)

dataset_sha256 = calculate_sha256(DATASET)

print(f"SHA-256: {dataset_sha256}")
print("Dataset validation passed.")

## Scientific Rule

The demo IDs are shuffled once using a fixed seed. D50 consists of the first 50 demos in this randomized order, D100 the first 100, and D200 the first 200. The exact same list is used for both RGB and Events.

In [ ]:
import h5py
import json
import random

DATASET = paths.raw / "lift_ph_raw.hdf5"

assert DATASET.exists(), f"Missing dataset: {DATASET}"

with h5py.File(DATASET, "r") as f:
    assert "data" in f, "Missing required HDF5 group: data"

    demo_ids = sorted(f["data"].keys())

    # Verify the expected number of demonstrations.
    assert len(demo_ids) == 200, (
        f"Expected 200 demonstrations, found {len(demo_ids)}."
    )

    # Verify that demonstration IDs are unique.
    assert len(demo_ids) == len(set(demo_ids)), (
        "Duplicate demonstration IDs detected."
    )

    lengths = {}
    action_shapes = {}
    state_shapes = {}

    for demo_id in demo_ids:
        demo = f["data"][demo_id]

        # Each demonstration must contain the recorded actions and states.
        assert "actions" in demo, (
            f"{demo_id}: missing 'actions' dataset."
        )
        assert "states" in demo, (
            f"{demo_id}: missing 'states' dataset."
        )

        actions = demo["actions"]
        states = demo["states"]

        # Store shapes for later inspection.
        action_shapes[demo_id] = tuple(actions.shape)
        state_shapes[demo_id] = tuple(states.shape)

        # Actions and states must have the same number of time samples.
        assert len(actions) == len(states), (
            f"{demo_id}: actions length {len(actions)} "
            f"!= states length {len(states)}."
        )

        # The recorded num_samples attribute should match the actual arrays.
        num_samples = int(demo.attrs["num_samples"])

        assert num_samples == len(actions), (
            f"{demo_id}: num_samples={num_samples} "
            f"but actions contain {len(actions)} samples."
        )

        lengths[demo_id] = num_samples

    total_control_samples = sum(lengths.values())

print("Dataset structure validation passed.")
print(f"Demonstrations: {len(demo_ids)}")
print(f"Total control samples: {total_control_samples}")
print(f"Example action shape: {action_shapes[demo_ids[0]]}")
print(f"Example state shape: {state_shapes[demo_ids[0]]}")

In [ ]:
import json
import random

# Dataset split seed is used exclusively for deterministic demo selection.
DATASET_SPLIT_SEED = 20260927

rng = random.Random(DATASET_SPLIT_SEED)

ordered = demo_ids.copy()

# Verify that the source demo IDs themselves are unique.
assert len(ordered) == len(set(ordered)), (
    "Duplicate demonstration IDs detected in the raw dataset."
)

rng.shuffle(ordered)

split_dir = paths.root / "configs" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

split_payloads = {}

for n in (50, 100, 200):
    assert len(ordered) >= n, (
        f"Cannot create D{n}: only {len(ordered)} demonstrations available."
    )

    split_payloads[n] = {
        "selection_seed": DATASET_SPLIT_SEED,
        "demo_ids": ordered[:n],
    }

    output_path = split_dir / f"d{n}.json"
    output_path.write_text(
        json.dumps(split_payloads[n], indent=2),
        encoding="utf-8",
    )

    print(f"Saved {output_path}")

# ------------------------------------------------------------------
# Explicit split validation
# ------------------------------------------------------------------

d50_ids = set(split_payloads[50]["demo_ids"])
d100_ids = set(split_payloads[100]["demo_ids"])
d200_ids = set(split_payloads[200]["demo_ids"])

# Exact cardinality checks.
assert len(d50_ids) == 50, f"D50 contains {len(d50_ids)} unique demos, expected 50."
assert len(d100_ids) == 100, f"D100 contains {len(d100_ids)} unique demos, expected 100."
assert len(d200_ids) == 200, f"D200 contains {len(d200_ids)} unique demos, expected 200."

# Nested-subset checks.
assert d50_ids.issubset(d100_ids), "D50 is not a subset of D100."
assert d100_ids.issubset(d200_ids), "D100 is not a subset of D200."

# No duplicates inside the ordered source list.
assert len(ordered) == 200, f"Expected 200 ordered demos, found {len(ordered)}."
assert len(d50_ids) == len(split_payloads[50]["demo_ids"])
assert len(d100_ids) == len(split_payloads[100]["demo_ids"])
assert len(d200_ids) == len(split_payloads[200]["demo_ids"])

print("\nNested split validation passed:")
print("D50  ⊂  D100  ⊂  D200")
print(f"D50  = {len(d50_ids)} demonstrations")
print(f"D100 = {len(d100_ids)} demonstrations")
print(f"D200 = {len(d200_ids)} demonstrations")
print(f"Dataset split seed = {DATASET_SPLIT_SEED}")

## Results Summary

At the end of the run, briefly summarize:

- What was actually executed?
- Which configuration and seed were used?
- Which files were created and at what path?
- What are the key metrics?
- What failed or remains open?
- What is the decision moving forward?

In [ ]:
from src.run_logging import write_notebook_summary

split_files = [
    split_dir / "d50.json",
    split_dir / "d100.json",
    split_dir / "d200.json",
]

summary = {
    "notebook": "01_Dataset_Audit_and_Splits.ipynb",
    "status": "completed",

    "configuration": {
        "task": "Lift",
        "dataset_type": "Proficient-Human raw",
        "dataset_version": "robosuite v1.5",
        "dataset_split_seed": DATASET_SPLIT_SEED,
        "nested_split_sizes": [50, 100, 200],
    },

    "outputs": [
        str(DATASET),
        *[str(path) for path in split_files],
    ],

    "metrics": {
        "number_of_demonstrations": len(demo_ids),
        "total_control_samples": int(total_control_samples),
        "dataset_size_mb": round(DATASET.stat().st_size / 1024**2, 2),
        "dataset_sha256": dataset_sha256,
        "d50_demonstrations": 50,
        "d100_demonstrations": 100,
        "d200_demonstrations": 200,
    },

    "validation_checks": {
        "expected_demo_count": 200,
        "demo_ids_unique": True,
        "actions_present_for_all_demos": True,
        "states_present_for_all_demos": True,
        "actions_states_lengths_consistent": True,
        "num_samples_matches_recorded_lengths": True,
        "d50_size_valid": True,
        "d100_size_valid": True,
        "d200_size_valid": True,
        "d50_subset_of_d100": True,
        "d100_subset_of_d200": True,
    },

    "observations": (
        "The Lift PH raw dataset passed the integrity and split validation "
        "checks implemented in this notebook. "
        "All 200 demonstration IDs are unique, each demonstration contains "
        "actions and states with consistent sample counts, and the recorded "
        "num_samples attributes match the actual trajectory lengths. "
        "D50 is a strict subset of D100, and D100 is a strict subset of D200. "
        "The same demonstration IDs will be used for RGB and Events."
    ),

    "decision": (
        "Use D50 for initial development and controlled RGB-versus-Events "
        "experiments. Keep D100 and D200 for later dataset-scaling experiments. "
        "The dataset split seed is reserved exclusively for deterministic split "
        "selection and is not a training seed."
    ),

    "next_step": (
        "Inspect the simulator metadata and replay one trajectory in MuJoCo "
        "using the fixed agentview and robot0_eye_in_hand cameras."
    ),
}

summary_path = write_notebook_summary(paths.summaries, summary)

print(f"Summary saved to: {summary_path}")
summary